# YOLO V5 — URL download + seven-class training (no Drive, no browser upload)

Use **one publicly accessible direct HTTPS download URL** for the original complete dataset ZIP `YOLO_V5_partial_annotation_audit320.zip`. Paste it into the `DATASET_URL` field below, then run all cells in Google Colab with a T4 GPU.

**Important:** The dataset has **not yet been uploaded to a public host**. Until you publish the ZIP as a GitHub Release asset or another direct-download location, no working URL can be supplied. This notebook does not magically host the dataset.

The dataset contains 2,093 training, 97 validation and 71 reserved test images. Labels include provisional annotations; the 1,252-worker audit was incomplete at the last checkpoint. The validation set lacks three class labels and its mAP cannot be described as full seven-class accuracy.


In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pyyaml" "pillow"

## 1. Download the full dataset directly by URL

Set `DATASET_URL` to an **actual** public direct-download URL for the 204-MiB ZIP (for example, a GitHub Release asset URL **after** the asset has been uploaded). This is not a GitHub repository page URL. No Google Drive or Colab browser upload is needed.


In [ ]:
from pathlib import Path
import hashlib, json, zipfile, shutil, os, urllib.request
from collections import Counter

# REQUIRED: paste the public DIRECT URL to the original complete archive once uploaded.
DATASET_URL = ''  # Example format only: https://github.com/OWNER/REPO/releases/download/TAG/YOLO_V5_partial_annotation_audit320.zip
EXPECTED_NAME = 'YOLO_V5_partial_annotation_audit320.zip'
EXPECTED_SHA256 = '4410e575f3797acca490c240fd58b85113aefb1261157c24c257b5dbeb2ecd9b'
BASE_ZIP = Path('/content') / EXPECTED_NAME
DATASET_DIR = Path('/content/v5_original_plus_v4')

if not DATASET_URL.startswith(('https://','http://')):
    raise ValueError('No dataset host is configured. First publish the 204-MiB ZIP, then paste its direct HTTPS download URL into DATASET_URL.')

if not BASE_ZIP.is_file():
    req = urllib.request.Request(DATASET_URL, headers={'User-Agent':'Colab-Research-Dataset/1.0'})
    try:
        with urllib.request.urlopen(req, timeout=90) as response, BASE_ZIP.open('wb') as output:
            print('Downloading from',response.geturl())
            while True:
                data = response.read(8*1024*1024)
                if not data: break
                output.write(data)
    except Exception:
        BASE_ZIP.unlink(missing_ok=True)
        raise

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

actual = sha256_file(BASE_ZIP)
assert actual == EXPECTED_SHA256, f'Wrong/corrupted ZIP, or URL returned an HTML page; actual SHA256={actual}'
print('Verified full dataset ZIP:', BASE_ZIP.name, f'({BASE_ZIP.stat().st_size/1024**2:.1f} MiB)')


## 2. Extract and validate

Checks ZIP integrity, class names, image/label pair counts and all manifest checksums. Does not modify any validation/test images.

In [ ]:
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

with zipfile.ZipFile(BASE_ZIP) as z:
    assert z.testzip() is None, 'ZIP CRC integrity failed'
    for name in z.namelist():
        path = Path(name.replace('\\', '/'))
        assert not path.is_absolute() and '..' not in path.parts, f'Unsafe ZIP member: {name}'
        assert path.parts and path.parts[0] == DATASET_DIR.name, f'Unexpected ZIP root: {name}'
    z.extractall('/content')

assert DATASET_DIR.is_dir()
with (DATASET_DIR / 'manifest.json').open(encoding='utf-8') as f:
    manifest = json.load(f)
assert Counter(x['split'] for x in manifest) == {'train':2093,'val':97,'test':71}

for item in manifest:
    for kind in ('image', 'label'):
        fp = DATASET_DIR / item[f'{kind}_path']
        assert fp.is_file(), f'Missing {kind}: {fp}'
        assert sha256_file(fp) == item[f'{kind}_sha256'], f'Checksum mismatch: {fp}'
    for line in (DATASET_DIR / item['label_path']).read_text().splitlines():
        if not line.strip():
            continue
        c,*coords=line.split()
        assert c in set(map(str,range(7))) and len(coords)==4
        xc,yc,w,h=map(float,coords)
        assert 0<w<=1 and 0<h<=1
        assert -0.00001<=xc-w/2<=xc+w/2<=1.00001
        assert -0.00001<=yc-h/2<=yc+h/2<=1.00001

import yaml
classes = ['food','plate','utensil','hinged_container','drink_carton','wrapper','bottle']
config = {'path':str(DATASET_DIR),'train':'images/train','val':'images/val',
          'test':'images/test','names':{i:s for i,s in enumerate(classes)}}
with (DATASET_DIR/'data.yaml').open('w') as f:
    yaml.safe_dump(config,f,sort_keys=False)

print('✅ Dataset integrity verified: 2,093 train / 97 validation / 71 test')
print('⚠️ Provisional annotations: 1,252 assigned originals still await individual worker review')
print('⚠️ Validation AP is not a valid seven-class accuracy estimate')


## 3. Train the unified seven-class model

This starts YOLO11s from pretrained weights. Uses **60 epochs, 800-pixel resolution, batch size 4, AdamW**. All outputs are stored under `/content/v5_training_runs`, not Google Drive.

A Colab runtime reset deletes local output files; **download the checkpoint before closing the session**.

In [ ]:
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4 GPU'
EPOCHS = 60
IMAGE_SIZE = 800
BATCH_SIZE = 4
RUN_NAME = 'V5_7class_audit320_provisional'

print('GPU:', torch.cuda.get_device_name(0))
print('⚠️ Training on provisional labels; this is NOT an independently verified benchmark.')
model = YOLO('yolo11s.pt')
train_results = model.train(
    data=str(DATASET_DIR/'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    optimizer='AdamW',
    lr0=0.0005,
    patience=20,
    workers=2,
    seed=57,
    deterministic=True,
    amp=True,
    cos_lr=True,
    degrees=7.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.05,
    close_mosaic=10,
    save_period=5,
    project='/content/v5_training_runs',
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
)
run_dir = Path(model.trainer.save_dir)
best_path = run_dir / 'weights' / 'best.pt'
assert best_path.exists(), f'Checkpoint missing: {best_path}'
print('✅ Finished:', best_path)


## 4. Download the trained weights

Run this cell after training. It downloads `best.pt` directly to your computer; **no Drive connection required**.

In [ ]:
from google.colab import files
assert best_path.is_file(), 'Run the training cell first.'
files.download(str(best_path))